# Learning Rate Search for EMG-Pose LSTM

This notebook finds the best learning rate for the LSTM model using two methods:

1. **LR Range Test** (Smith 1-cycle finder) — sweep LR from very small to very large over one epoch, plot loss vs LR to find the steepest descent region
2. **Grid Search** — train for N epochs at each candidate LR, compare final test MAE

Both methods import directly from the repo's `.py` files.

## 1. Setup

In [ ]:
!pip install -q h5py scipy pandas matplotlib

import os

REPO_DIR = "Erdos_dl_emg-pose"
if not os.path.exists(REPO_DIR):
    # UPDATE this URL to your actual repo
    !git clone https://github.com/YOUR_USERNAME/Erdos_dl_emg-pose.git
else:
    print(f"Repo already cloned at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Download mini dataset
DATA_DIR = "emg2pose_dataset_mini"
if not os.path.exists(DATA_DIR):
    print("Downloading mini dataset (~600 MiB)...")
    !curl -sL "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_dataset_mini.tar" -o emg2pose_dataset_mini.tar
    !tar -xf emg2pose_dataset_mini.tar
    !rm emg2pose_dataset_mini.tar
    print("Done.")
else:
    print(f"Dataset already at {DATA_DIR}")

print(f"HDF5 files: {len([f for f in os.listdir(DATA_DIR) if f.endswith('.hdf5')])}")

In [ ]:
import copy
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from load_data import get_dataloaders, BATCH_SIZE
from model import EMGPoseLSTM
from train import train_one_epoch, evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load data once — reused by all experiments
loaders = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    use_val=False,
)

print(f"\n{'Split':<8} {'Batches':>8} {'Samples':>10}")
for name, loader in loaders.items():
    print(f"{name:<8} {len(loader):>8} {len(loader.dataset):>10}")

## 2. LR Range Test (Smith finder)

Sweep LR exponentially from `lr_min` to `lr_max` over one epoch. Track the loss at each step. The optimal LR is in the region where loss decreases fastest (steepest slope), typically ~1 order of magnitude before the minimum.

In [ ]:
def lr_range_test(loaders, hidden_size=256, num_layers=2,
                  lr_min=1e-6, lr_max=1.0, num_steps=None,
                  max_grad_norm=1.0, smooth_factor=0.05):
    """Run an LR range test over the training set.

    Args:
        loaders: Dict with 'train' DataLoader.
        lr_min: Starting learning rate.
        lr_max: Ending learning rate.
        num_steps: Number of LR steps (default = num batches).
        smooth_factor: Exponential smoothing factor for loss.

    Returns:
        lrs: List of learning rates tested.
        losses: List of smoothed losses at each step.
    """
    model = EMGPoseLSTM(hidden_size=hidden_size, num_layers=num_layers).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr_min)

    train_loader = loaders["train"]
    if num_steps is None:
        num_steps = len(train_loader)

    # Exponential LR schedule: lr_min -> lr_max
    gamma = (lr_max / lr_min) ** (1.0 / num_steps)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

    lrs, losses = [], []
    smoothed_loss = None
    best_loss = float("inf")
    model.train()

    for i, batch in enumerate(train_loader):
        if i >= num_steps:
            break

        emg = batch["emg"].to(device)
        targets = batch["joint_angles"].to(device)
        mask = batch["no_ik_failure"].to(device)

        preds = model(emg)
        mask_exp = mask.unsqueeze(1).expand_as(preds)
        if not mask_exp.any():
            continue

        loss = F.l1_loss(preds[mask_exp], targets[mask_exp])

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        current_lr = optimizer.param_groups[0]["lr"]
        loss_val = loss.item()

        # Exponential smoothing
        if smoothed_loss is None:
            smoothed_loss = loss_val
        else:
            smoothed_loss = smooth_factor * loss_val + (1 - smooth_factor) * smoothed_loss

        lrs.append(current_lr)
        losses.append(smoothed_loss)

        # Stop if loss diverges (> 4x best)
        if smoothed_loss < best_loss:
            best_loss = smoothed_loss
        if smoothed_loss > 4 * best_loss:
            print(f"  Stopping at step {i}: loss diverged (lr={current_lr:.2e})")
            break

        scheduler.step()

    print(f"  LR range test: {len(lrs)} steps, lr {lrs[0]:.2e} -> {lrs[-1]:.2e}")
    return lrs, losses

In [ ]:
# Run LR range test
lrs, losses = lr_range_test(loaders, lr_min=1e-6, lr_max=1.0)

In [ ]:
def plot_lr_range_test(lrs, losses):
    """Plot loss vs LR from range test. Annotate suggested LR."""
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(lrs, losses, "b-", linewidth=1.5)
    ax.set_xscale("log")
    ax.set_xlabel("Learning Rate")
    ax.set_ylabel("Smoothed Loss (MAE)")
    ax.set_title("LR Range Test")

    # Find steepest descent: LR where gradient of loss is most negative
    log_lrs = np.log10(lrs)
    grad = np.gradient(losses, log_lrs)
    min_grad_idx = np.argmin(grad)
    suggested_lr = lrs[min_grad_idx]

    ax.axvline(suggested_lr, color="r", linestyle="--", alpha=0.7,
               label=f"Suggested LR: {suggested_lr:.2e}")

    # Also mark the loss minimum
    min_loss_idx = np.argmin(losses)
    ax.axvline(lrs[min_loss_idx], color="g", linestyle=":", alpha=0.7,
               label=f"Min loss LR: {lrs[min_loss_idx]:.2e}")

    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

    print(f"\nSuggested LR (steepest descent): {suggested_lr:.2e}")
    print(f"Min loss LR: {lrs[min_loss_idx]:.2e}")
    print(f"Recommended range to grid search: [{suggested_lr/3:.2e}, {lrs[min_loss_idx]:.2e}]")

    return suggested_lr

suggested_lr = plot_lr_range_test(lrs, losses)

## 3. Grid Search Over Learning Rates

Train the model from scratch at each candidate LR for a fixed number of epochs. Compare final test MAE to pick the winner.

In [ ]:
def train_with_lr(loaders, lr, epochs=15, hidden_size=256, num_layers=2):
    """Train a fresh model at a given LR. Returns per-epoch history."""
    model = EMGPoseLSTM(hidden_size=hidden_size, num_layers=num_layers).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "test_mae": []}
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, loaders["train"], optimizer, device)
        test_mae = evaluate(model, loaders["test"], device)
        history["train_loss"].append(train_loss)
        history["test_mae"].append(test_mae)

    return history

In [ ]:
# Candidate learning rates
LR_CANDIDATES = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
GRID_EPOCHS = 15  # epochs per candidate

print(f"Grid search: {len(LR_CANDIDATES)} LRs x {GRID_EPOCHS} epochs each\n")

results = {}
for lr in LR_CANDIDATES:
    print(f"  lr={lr:.1e} ... ", end="", flush=True)
    history = train_with_lr(loaders, lr=lr, epochs=GRID_EPOCHS)
    results[lr] = history
    best_test = min(history["test_mae"])
    final_test = history["test_mae"][-1]
    print(f"best_test_mae={best_test:.4f}  final_test_mae={final_test:.4f}")

print("\nGrid search complete.")

In [ ]:
# Plot all LR curves together
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.viridis(np.linspace(0, 0.9, len(LR_CANDIDATES)))

# Left: train loss
for (lr, hist), color in zip(results.items(), colors):
    epochs = range(1, len(hist["train_loss"]) + 1)
    ax1.plot(epochs, hist["train_loss"], color=color, label=f"lr={lr:.1e}")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss (MAE)")
ax1.set_title("Train Loss by Learning Rate")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Right: test MAE
for (lr, hist), color in zip(results.items(), colors):
    epochs = range(1, len(hist["test_mae"]) + 1)
    ax2.plot(epochs, hist["test_mae"], color=color, label=f"lr={lr:.1e}")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Test MAE")
ax2.set_title("Test MAE by Learning Rate")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# Summary bar chart: best test MAE per LR
best_maes = {lr: min(h["test_mae"]) for lr, h in results.items()}
best_lr = min(best_maes, key=best_maes.get)

fig, ax = plt.subplots(figsize=(10, 5))
lr_labels = [f"{lr:.1e}" for lr in best_maes.keys()]
mae_vals = list(best_maes.values())
bar_colors = ["green" if lr == best_lr else "steelblue" for lr in best_maes.keys()]

ax.bar(lr_labels, mae_vals, color=bar_colors)
ax.set_xlabel("Learning Rate")
ax.set_ylabel("Best Test MAE")
ax.set_title("Best Test MAE by Learning Rate")
ax.grid(axis="y", alpha=0.3)

# Annotate best
best_idx = list(best_maes.keys()).index(best_lr)
ax.annotate(f"Best: {best_maes[best_lr]:.4f}",
            xy=(best_idx, best_maes[best_lr]),
            xytext=(best_idx, best_maes[best_lr] * 0.95),
            ha="center", fontsize=11, fontweight="bold", color="green")

fig.tight_layout()
plt.show()

print(f"\nBest learning rate: {best_lr:.1e} (test MAE = {best_maes[best_lr]:.4f})")

## 4. Summary

In [ ]:
print("=" * 50)
print("Learning Rate Search Results")
print("=" * 50)
print(f"\nLR Range Test suggested LR:  {suggested_lr:.2e}")
print(f"Grid Search best LR:         {best_lr:.1e}")
print(f"Grid Search best test MAE:   {best_maes[best_lr]:.4f}")
print(f"\nAll grid search results:")
print(f"{'LR':>10}  {'Best Test MAE':>14}  {'Final Test MAE':>15}")
print("-" * 45)
for lr, hist in sorted(results.items()):
    marker = " <-- best" if lr == best_lr else ""
    print(f"{lr:>10.1e}  {min(hist['test_mae']):>14.4f}  {hist['test_mae'][-1]:>15.4f}{marker}")

print(f"\nRecommendation: Use lr={best_lr:.1e} for training.")

In [ ]:
# Save results
output = {
    "suggested_lr_range_test": suggested_lr,
    "best_lr_grid_search": best_lr,
    "grid_results": {
        str(lr): {
            "best_test_mae": min(h["test_mae"]),
            "final_test_mae": h["test_mae"][-1],
            "train_loss": h["train_loss"],
            "test_mae": h["test_mae"],
        }
        for lr, h in results.items()
    },
}

out_path = Path("checkpoints/lr_search_results.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Saved results to {out_path}")

# Uncomment to download from Colab:
# from google.colab import files
# files.download(str(out_path))